In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# ---- Set your file path here ----
DATA_PATH = Path("sp500_panel.csv")  # Update path if needed

# Runtime settings
SAMPLE_ROWS = 200_000
CHUNKSIZE = 200_000
TARGET_COL = "y_return"

print("Using file:", DATA_PATH)

In [ ]:
# missing value check
n_rows = len(df)

missing_count = df.isna().sum()
missing_pct = (missing_count / n_rows * 100).round(2)

missing_table = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending=False)

display(missing_table.head(20))

In [ ]:
#duplicate check
print("Duplicate full rows:", df.duplicated().sum())

date_col = "date" if "date" in df.columns else None
ticker_col = "ticker" if "ticker" in df.columns else None

if date_col and ticker_col:
    key_duplicates = df.duplicated(subset=[ticker_col, date_col]).sum()
    print(f"Duplicate ({ticker_col}, {date_col}) keys:", key_duplicates)

In [ ]:
# structure anlaysis
if ticker_col:
    print("Unique tickers:", df[ticker_col].nunique())

if date_col:
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    print("Date range:", df[date_col].min(), "to", df[date_col].max())

if ticker_col:
    display(df.groupby(ticker_col).size().describe())

In [ ]:
# checking skewness of the data
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

skew_series = df[numeric_cols].apply(
    lambda s: pd.to_numeric(s, errors='coerce').dropna().skew()
)

skew_table = skew_series.to_frame("skewness").sort_values("skewness", ascending=False)

def skew_label(v):
    if pd.isna(v): return "NA"
    if v > 1: return "Highly right-skewed"
    if v > 0.5: return "Moderately right-skewed"
    if v < -1: return "Highly left-skewed"
    if v < -0.5: return "Moderately left-skewed"
    return "Approximately symmetric"

skew_table["interpretation"] = skew_table["skewness"].apply(skew_label)

display(skew_table)

In [ ]:
#outlier detection
outlier_rows = []

for c in numeric_cols:
    s = pd.to_numeric(df[c], errors='coerce').dropna()
    if len(s) < 5:
        continue
        
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    
    out_count = ((s < lo) | (s > hi)).sum()
    out_pct = 100 * out_count / len(s)
    
    outlier_rows.append([c, int(out_count), round(out_pct,2), lo, hi])

outlier_df = pd.DataFrame(
    outlier_rows,
    columns=["feature", "outlier_count", "outlier_pct", "lower_bound", "upper_bound"]
).sort_values("outlier_pct", ascending=False)

display(outlier_df)

In [ ]:
#correlation analysis (chunked so memory safe)

preview = pd.read_csv(DATA_PATH, nrows=5000)
preview.columns = preview.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)

candidate_numeric = preview.select_dtypes(include=[np.number]).columns.tolist()

corr_sample_df = pd.DataFrame(columns=candidate_numeric)
max_sample_rows = 200_000
rows_seen = 0
rng = np.random.default_rng(42)

for chunk in pd.read_csv(DATA_PATH, chunksize=CHUNKSIZE, usecols=candidate_numeric):
    
    chunk.columns = chunk.columns.str.strip().str.lower().str.replace(" ", "_", regex=False)
    chunk = chunk.dropna(how="all")
    
    chunk_n = len(chunk)
    rows_seen += chunk_n
    
    space_left = max_sample_rows - len(corr_sample_df)
    
    if space_left > 0:
        take_n = min(space_left, chunk_n)
        corr_sample_df = pd.concat(
            [corr_sample_df, chunk.iloc[:take_n]],
            ignore_index=True
        )
        chunk = chunk.iloc[take_n:]
    
    if not chunk.empty and len(corr_sample_df) > 0:
        p = max_sample_rows / rows_seen
        keep_mask = rng.random(len(chunk)) < p
        candidates = chunk.loc[keep_mask]
        
        if not candidates.empty:
            replace_idx = rng.integers(0, len(corr_sample_df), size=len(candidates))
            corr_sample_df.iloc[replace_idx] = candidates.values

print("Rows scanned:", rows_seen)
print("Rows kept:", len(corr_sample_df))

corr = corr_sample_df.corr()
display(corr.round(3))


In [ ]:
#heatmap

plt.figure(figsize=(10,8))
plt.imshow(corr, aspect='auto')
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Matrix (Chunked Sample)")
plt.tight_layout()
plt.show()


Overall Observations for part 1:

1. Basic Info about data -:

total data entries : 1218220

columns: 16

data types: float64(12), int64(2), object(2)

2. Important data analysis values:

count 501.000000

mean 2431.576846

std 336.325100

min 34.000000

25% 2510.000000

50% 2510.000000

75% 2510.000000

max 2510.000000

// Also a quick check to make sure null values were removed during pre-processing.

3. Graphical Distributions:

Normal closing graph did not help much, also made a log graph, appears as a gaussian curve.

4. Top 10 countries:

Nvidia, Tesla, Apple, Netflix, Amazon, AMD, BAC, PLTR, F, T

5. Avg & Individual stocks

 Average stock trends for all companies an few individual companies.

 a. Nvidia had a flat growth in the initial years and started picking up in 2023.

 b. Tesla stocks started slowly piking in 2019 but keep rigourously fluctuating between pikes and lows.

 c. Other stocks like Apple, Microsoft, Google and Amazon all started picking up in 2019 and are still rising with microsoft having the hightest peak in 2025.

 d. General observation is that except Nvidia, every other company started peaking in 2019 but NVidia was abit later in 2021.  

The EDA shows that the dataset is generally suitable for modeling, with clear structure across core price/volume features. Several numeric variables are visibly right-skewed and contain extreme values, indicating that robust preprocessing (such as scaling, winsorization, and log transforms) is important before training. Correlation patterns suggest meaningful relationships among market variables while preserving sufficient distinct information for multi-feature models. The outlier analysis also confirms that volatility-heavy periods are present, which is consistent with the project’s regime-based framing. These results support using market-regime signals together with stock-level engineered features rather than relying on stock-only inputs. Overall, the EDA findings provide a solid foundation for moving into model training, hyperparameter tuning, and comparative evaluation across baseline and regime-aware approaches.